<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/Representacao_Interna_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Representação Interna em uma Rede Neural**

## Treinamento dos Embeddings  

In [91]:
%%script echo off
'''
Token:        "king"
↓
ID:           12345
↓
Embedding inicial: vetor aleatório [0.002, -0.045, ..., 0.013]
↓
Treinamento
↓
Embedding final: vetor aprendido [0.181, -0.407, ..., 0.233]
'''

off


In [92]:
import numpy as np
import pandas as pd
np.random.seed(42)

# ---------------------------
# 1. Mini vocabulário
# ---------------------------
vocab = ["king", "queen", "man", "woman", "apple", "orange"]
vocab_size = len(vocab)
word_to_id = {w: i for i, w in enumerate(vocab)}

# ---------------------------
# 2. Pares (central, contexto)
# ---------------------------
# Em um corpus real, isso viria da coocorrência em janelas de texto
pairs = [
    ("king", "queen"), ("queen", "king"),
    ("man", "woman"), ("woman", "man"),
    ("king", "man"), ("queen", "woman"),
    ("apple", "orange"), ("orange", "apple")
]

# ---------------------------
# 3. Inicialização dos embeddings
# ---------------------------
embedding_dim = 4  # pequeno só para visualização
W_in = np.random.randn(vocab_size, embedding_dim) * 0.01   # embeddings de entrada
W_out = np.random.randn(vocab_size, embedding_dim) * 0.01  # embeddings de saída

# ---------------------------
# 4. Função utilitária softmax
# ---------------------------
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

# ---------------------------
# *. Antes do treinamento
# ---------------------------
print('\n*** Antes do Treinamento...\n')

for word in vocab:
  u = np.dot(W_out, W_in[word_to_id[word]])
  y_pred = softmax(u).argmax()

  print(f"word {word} → {vocab[y_pred]}")

# ---------------------------
# 5. Treinamento (Skip-gram simplificado)
# ---------------------------
print('\n*** Treinamento...\n')

lr = 0.05
epochs = 10000

for epoch in range(epochs):
    loss = 0
    for center, context in pairs:
        c_id = word_to_id[center]
        o_id = word_to_id[context]

        # Forward
        h = W_in[c_id]                  # embedding do centro
        u = np.dot(W_out, h)            # pontua todos os contextos possíveis
        y_pred = softmax(u)             # distribuição de probabilidade
        loss -= np.log(y_pred[o_id] + 1e-9)  # perda (cross-entropy)

        # Backprop simplificado
        y_pred[o_id] -= 1  # derivada da loss w.r.t. logits
        dW_out = np.outer(y_pred, h)
        dW_in = np.dot(W_out.T, y_pred)

        # Atualização
        W_in[c_id] -= lr * dW_in
        W_out -= lr * dW_out

    if epoch % 1000 == 0:
        print(f"Época {epoch:4d} | Loss média: {loss/len(pairs):.4f}")

# ---------------------------
# 6. Resultados: embeddings finais
# ---------------------------
embeddings = W_in / np.linalg.norm(W_in, axis=1, keepdims=True)

# ---------------------------
# *. Depois do treinamento
# ---------------------------
print('\n*** Depois do Treinamento...\n')

for word in vocab:
  u = np.dot(W_out, W_in[word_to_id[word]])
  y_pred = softmax(u).argmax()

  print(f"word {word} → {vocab[y_pred]}")

print()


*** Antes do Treinamento...

word king → queen
word queen → queen
word man → king
word woman → woman
word apple → king
word orange → man

*** Treinamento...

Época    0 | Loss média: 1.7918
Época 1000 | Loss média: 0.3806
Época 2000 | Loss média: 0.3721
Época 3000 | Loss média: 0.3681
Época 4000 | Loss média: 0.3657
Época 5000 | Loss média: 0.3639
Época 6000 | Loss média: 0.3626
Época 7000 | Loss média: 0.3616
Época 8000 | Loss média: 0.3608
Época 9000 | Loss média: 0.3601

*** Depois do Treinamento...

word king → man
word queen → woman
word man → woman
word woman → man
word apple → orange
word orange → apple



## Representação dos termos

In [93]:
import pandas as pd

df = pd.DataFrame(W_in)
df.index = vocab

df

,0,1,2,3
king,0.389585,-0.505350,0.662851,0.496444
queen,-0.495768,0.376409,-0.518364,0.660336
man,-2.447523,-2.871167,-4.016188,-0.458079
woman,1.245511,-5.059827,-0.418861,-1.790157
apple,-1.929001,0.599624,0.754820,-2.168024
orange,2.263415,1.404447,-0.313317,-1.269506


## Similaridade Coseno

$$ \text{similaridade}_{\cos}(a,b) = \frac{a.b}{|a|.|b|} $$

In [94]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [95]:
# Create a new row as a DataFrame
df = pd.DataFrame(W_in)
df.index = vocab

new_row_data = df.loc['king'].values - np.random.rand(len(df.loc['king'].values))*0.1
new_row_df = pd.DataFrame([new_row_data], index=['king~'], columns=df.columns)

# Concatenate the new row to the existing DataFrame
df = pd.concat([df, new_row_df])

df['similarity to First'] = df.apply(lambda row: cosine_similarity(row.values, df.loc['king~'].values), axis=1)
df = df.sort_values(by='similarity to First', ascending=False)

display(df)

,0,1,2,3,similarity to First
king~,0.361491,-0.559620,0.648758,0.416224,1.000000
king,0.389585,-0.505350,0.662851,0.496444,0.995466
woman,1.245511,-5.059827,-0.418861,-1.790157,0.402330
orange,2.263415,1.404447,-0.313317,-1.269506,-0.231349
man,-2.447523,-2.871167,-4.016188,-0.458079,-0.368198
queen,-0.495768,0.376409,-0.518364,0.660336,-0.423862
apple,-1.929001,0.599624,0.754820,-2.168024,-0.464004


## Similaridade Produto Vetorial

$$ \text{similaridade}_{dot}(a,b) = a.b$$

In [96]:
def dot_product(a, b):
    return np.dot(a, b)

In [97]:
df = pd.DataFrame(W_in)
df.index = vocab
df = df.apply(lambda x: x / np.linalg.norm(x), axis=1)

new_row_data = df.loc['king'].values - np.random.rand(len(df.loc['king'].values))*0.1
new_row_data = new_row_data / np.linalg.norm(new_row_data)
new_row_df = pd.DataFrame([new_row_data], index=['king~'], columns=df.columns)

# Concatenate the new row to the existing DataFrame
df = pd.concat([df, new_row_df])

df['similarity to First'] = df.apply(lambda row: cosine_similarity(row.values, df.loc['king~'].values), axis=1)
df = df.sort_values(by='similarity to First', ascending=False)

df['dot_product'] = df.apply(lambda row: dot_product(row.values, df.loc['king~'].values), axis=1)

display(df)

,0,1,2,3,similarity to First,dot_product
king~,0.367177,-0.585232,0.559834,0.457461,1.000000,2.000000
king,0.372645,-0.483377,0.634029,0.474857,0.991894,1.983788
woman,0.225404,-0.915691,-0.075803,-0.323970,0.428015,0.856030
orange,0.762766,0.473296,-0.105587,-0.427821,-0.251740,-0.503481
man,-0.442643,-0.519261,-0.726342,-0.082845,-0.303169,-0.606338
queen,-0.474378,0.360169,-0.495999,0.631845,-0.373595,-0.747191
apple,-0.630828,0.196091,0.246844,-0.708994,-0.532530,-1.065060


## Embedding de um 'Texto'

A rigor cada termo/token tem uma representação de tamanho $n$ e havendo um documento $d_i$ teríamos vetores de tamanho $termos(d_i) \times m$, havendo tamanhos diferentes para cada documento $i$. Entradas de tamanho variável não podem ser empregadas em modelos tradicionais de ML. Uma forma de resolvermos isso é fazermos algum tipo de média ou soma dos vetores dos termos do documento, mantendo assim, a dimensão $n$.

$$ \text{Embedding}(d_i) = v(d_i) = \sum_{t \in d_i} v(t) $$

In [98]:
df = pd.DataFrame(W_in)
df.index = vocab
df = df.apply(lambda x: x / np.linalg.norm(x), axis=1)

new_row_data = df.loc['king'].values + df.loc['queen'].values
new_row_data = new_row_data / np.linalg.norm(new_row_data)
new_row_df = pd.DataFrame([new_row_data], index=['king+queen'], columns=df.columns)

# Concatenate the new row to the existing DataFrame
df = pd.concat([df, new_row_df])

df['dot_product'] = df.apply(lambda row: dot_product(row.values, df.loc['king+queen'].values), axis=1)
df = df.sort_values(by='dot_product', ascending=False)

display(df)

,0,1,2,3,dot_product
king+queen,-0.090296,-0.109356,0.122512,0.982282,1.000000
queen,-0.474378,0.360169,-0.495999,0.631845,0.563332
king,0.372645,-0.483377,0.634029,0.474857,0.563332
man,-0.442643,-0.519261,-0.726342,-0.082845,-0.073610
woman,0.225404,-0.915691,-0.075803,-0.323970,-0.247733
orange,0.762766,0.473296,-0.105587,-0.427821,-0.553809
apple,-0.630828,0.196091,0.246844,-0.708994,-0.630674
